In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC, SVR
import seaborn as sns
import os
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from tqdm import tqdm
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report, roc_curve, \
RocCurveDisplay, roc_auc_score, r2_score, mean_absolute_error
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import log_loss, f1_score
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.naive_bayes import BernoulliNB, GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import VotingRegressor, BaggingClassifier, BaggingRegressor, RandomForestClassifier, RandomForestRegressor, StackingClassifier
from sklearn.linear_model import ridge_regression, ElasticNet
from sklearn.linear_model import Ridge


In [14]:
df = pd.read_csv('../Cases/Glass_Identification/Glass.csv')
df

,RI,Na,Mg,Al,Si,K,Ca,Ba,Fe,Type
0,1.52101,13.64,4.49,1.10,71.78,0.06,8.75,0.00,0.0,building_windows_float_processed
1,1.51761,13.89,3.60,1.36,72.73,0.48,7.83,0.00,0.0,building_windows_float_processed
2,1.51618,13.53,3.55,1.54,72.99,0.39,7.78,0.00,0.0,building_windows_float_processed
3,1.51766,13.21,3.69,1.29,72.61,0.57,8.22,0.00,0.0,building_windows_float_processed
4,1.51742,13.27,3.62,1.24,73.08,0.55,8.07,0.00,0.0,building_windows_float_processed
...,...,...,...,...,...,...,...,...,...,...
209,1.51623,14.14,0.00,2.88,72.61,0.08,9.18,1.06,0.0,headlamps
210,1.51685,14.92,0.00,1.99,73.06,0.00,8.40,1.59,0.0,headlamps
211,1.52065,14.36,0.00,2.02,73.42,0.00,8.44,1.64,0.0,headlamps
212,1.51651,14.38,0.00,1.94,73.61,0.00,8.48,1.57,0.0,headlamps


In [15]:
x,y = df.drop('Type', axis = 1),df['Type']
x_train, x_test , y_train, y_test = train_test_split(x,y, random_state=25, test_size=0.3)

Scaling

In [16]:
scaler = StandardScaler()

In [17]:
import warnings
warnings.filterwarnings('ignore')

In [18]:
lr = LogisticRegression()
svm = SVC(kernel='linear')
pipe_svm = Pipeline([('Scaling', scaler), ("SVM", svm)])
dtc = DecisionTreeClassifier(random_state=25)
rf = RandomForestClassifier(random_state=25)
stack = StackingClassifier(estimators = [('LR', lr),('SVM', pipe_svm),('Tree', dtc)],final_estimator=rf)

stack.fit(x_train, y_train)
y_pred = stack.predict(x_test)

print(f1_score(y_test, y_pred, average='macro'))

0.5314405452703325


Checking individual f1 scores of every model

In [19]:
models = [lr, svm, rf, dtc]

for i in tqdm(models):
    
    model = i

    model.fit(x_train, y_train)
    y_pred = model.predict(x_test)
    print(f'{model} --> {f1_score(y_test, y_pred, average = 'macro')}')

  0%|          | 0/4 [00:00<?, ?it/s]

LogisticRegression() --> 0.5182539682539683
SVC(kernel='linear') --> 0.6132933960918555


100%|██████████| 4/4 [00:00<00:00, 16.23it/s]

RandomForestClassifier(random_state=25) --> 0.7047381711855395
DecisionTreeClassifier(random_state=25) --> 0.5557960557960557


using passthrough

In [20]:
stack = StackingClassifier(estimators = [('LR', lr),('SVM', pipe_svm),('Tree', dtc)],
                           final_estimator=rf,
                           passthrough=True)

stack.fit(x_train, y_train)
y_pred = stack.predict(x_test)

print(f1_score(y_test, y_pred, average='macro'))

0.602287788543205


using xgboost and LGB models as final estimators

In [22]:
import xgboost as xgb
import lightgbm as lgb


xgb = xgb.XGBClassifier(random_state = 25)

stack = StackingClassifier(estimators = [('LR', lr),('SVM', pipe_svm),('Tree', dtc)],
                           final_estimator=xgb,
                           passthrough=True)

stack.fit(x_train, y_train)
y_pred = stack.predict(x_test)

print(f1_score(y_test, y_pred, average='macro'))

0.6700071399428804


In [24]:
lgb = lgb.LGBMClassifier(random_state = 25, verbose = -1)

stack = StackingClassifier(estimators = [('LR', lr),('SVM', pipe_svm),('Tree', dtc)],
                           final_estimator=lgb,
                           passthrough=True)

stack.fit(x_train, y_train)
y_pred = stack.predict(x_test)

print(f1_score(y_test, y_pred, average='macro'))

0.5907301898638936
